In [ ]:
import os
import re
import shutil
from collections import defaultdict

In [ ]:
# =========================
# Source directories
# =========================
src_image_dir = "/root/nnunet/data/Breast_duke_data"
src_label_dir = "/root/nnunet/data/Breast_duke_label"

# =========================
# Destination directories
# =========================
imagesTr_dir = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/imagesTr"
labelsTr_dir = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/labelsTr"
imagesTs_dir = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/imagesTs"

# Create destination folders if they do not exist
os.makedirs(imagesTr_dir, exist_ok=True)
os.makedirs(labelsTr_dir, exist_ok=True)
os.makedirs(imagesTs_dir, exist_ok=True)

# =========================
# Regex patterns
# =========================
image_pattern = re.compile(r"^Breast_MRI_(\d{3})_(\d{2})\.nii\.gz$")
label_pattern = re.compile(r"^Breast_MRI_(\d{3})_label\.nii\.gz$")

# =========================
# Collect image files by case
# =========================
case_to_images = defaultdict(list)

for fname in os.listdir(src_image_dir):
    match = image_pattern.match(fname)
    if match:
        case_id, component_id = match.groups()
        case_to_images[case_id].append((int(component_id), fname))

# =========================
# Collect label files by case
# =========================
case_to_label = {}

for fname in os.listdir(src_label_dir):
    match = label_pattern.match(fname)
    if match:
        case_id = match.group(1)
        case_to_label[case_id] = fname

# =========================
# Copy data into nnUNet format
# =========================
copied_cases = 0
skipped_cases = []
component_count_summary = defaultdict(int)

for case_id in sorted(case_to_images.keys()):
    if case_id not in case_to_label:
        skipped_cases.append(case_id)
        print(f"[WARNING] Missing label for case {case_id}, skipping.")
        continue

    # Sort image components, e.g. 00, 01, 02, 03
    image_list = sorted(case_to_images[case_id], key=lambda x: x[0])

    # nnUNet case name
    nnunet_case_id = f"BreastMRI{case_id}"

    # Copy image components
    for new_channel_idx, (original_component_idx, src_fname) in enumerate(image_list):
        src_path = os.path.join(src_image_dir, src_fname)
        dst_fname = f"{nnunet_case_id}_{new_channel_idx:04d}.nii.gz"
        dst_path = os.path.join(imagesTr_dir, dst_fname)

        shutil.copy2(src_path, dst_path)
        print(f"[IMAGE] {src_fname}  -->  {dst_fname}")

    # Copy label
    src_label_path = os.path.join(src_label_dir, case_to_label[case_id])
    dst_label_fname = f"{nnunet_case_id}.nii.gz"
    dst_label_path = os.path.join(labelsTr_dir, dst_label_fname)

    shutil.copy2(src_label_path, dst_label_path)
    print(f"[LABEL] {case_to_label[case_id]}  -->  {dst_label_fname}")

    copied_cases += 1
    component_count_summary[len(image_list)] += 1

print("\n" + "=" * 60)
print("Dataset organization complete.")
print(f"Copied cases: {copied_cases}")
print(f"Skipped cases (missing label): {len(skipped_cases)}")

if skipped_cases:
    print("Skipped case IDs:", ", ".join(skipped_cases))

print("\nComponent count summary per case:")
for n_comp, n_cases in sorted(component_count_summary.items()):
    print(f"  {n_cases} case(s) with {n_comp} component(s)")

print("=" * 60)

In [ ]:
import SimpleITK as sitk
import numpy as np

path = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/labelsTr/BreastMRI246.nii.gz"

img = sitk.ReadImage(path)
arr = sitk.GetArrayFromImage(img)

print("Unique labels:", np.unique(arr))
print("dtype:", arr.dtype)

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
# nnUnet training looks for binary label mask, so we convert all labels to binary

labels_dir = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/labelsTr"

for fname in sorted(os.listdir(labels_dir)):
    if not fname.endswith(".nii.gz"):
        continue

    path = os.path.join(labels_dir, fname)
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)

    unique_before = np.unique(arr)

    # Merge all nonzero labels into 1
    arr_bin = (arr > 0).astype(np.uint8)

    out = sitk.GetImageFromArray(arr_bin)
    out.CopyInformation(img)
    sitk.WriteImage(out, path)

    unique_after = np.unique(arr_bin)
    print(f"{fname}: {unique_before} -> {unique_after}")

In [ ]:
import os
input_dir = "/root/breast_seg/test_images"
output_dir = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/test_labels/predictions"
dataset_id = "Dataset001_BreastTumor"
config = "3d_fullres"
fold = 4

cmd = f"""
nnUNetv2_predict \
-i {input_dir} \
-o {output_dir} \
-d {dataset_id} \
-c {config} \
-f {fold} \
-chk checkpoint_best.pth
"""
print("Running:\n", cmd)
os.system(cmd)

In [1]:
import os
import shutil
from pathlib import Path

def predict_single_image(
    image_path,
    output_mask_path,
    dataset_id=1,
    config="3d_fullres",
    fold=4,
):
    tmp_input = "/tmp/nnunet_single_input"
    tmp_output = "/tmp/nnunet_single_output"
    shutil.rmtree(tmp_input, ignore_errors=True)
    shutil.rmtree(tmp_output, ignore_errors=True)
    os.makedirs(tmp_input, exist_ok=True)
    os.makedirs(tmp_output, exist_ok=True)

    # nnU-Net requires naming like: case_XXX_0000.nii.gz
    tmp_image_name = "case_000_0000.nii.gz"
    tmp_image_path = os.path.join(tmp_input, tmp_image_name)
    shutil.copy(image_path, tmp_image_path)

    # run prediction
    cmd = f"""
    nnUNetv2_predict \
    -i {tmp_input} \
    -o {tmp_output} \
    -d {dataset_id} \
    -c {config} \
    -f {fold} \
    -chk checkpoint_best.pth
    """
    print("Running:\n", cmd)
    os.system(cmd)

    pred_file = os.path.join(tmp_output, "case_000.nii.gz")

    if not os.path.exists(pred_file):
        raise FileNotFoundError("Prediction not generated!")

    shutil.move(pred_file, output_mask_path)

    print(f"Prediction saved at: {output_mask_path}")

In [2]:
import os

os.environ["nnUNet_raw"] = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/root/breast_seg/nnUNet_Frame/dataset/nnUNet_results"

print("nnUNet_raw =", os.environ.get("nnUNet_raw"))
print("nnUNet_preprocessed =", os.environ.get("nnUNet_preprocessed"))
print("nnUNet_results =", os.environ.get("nnUNet_results"))

nnUNet_raw = /root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw
nnUNet_preprocessed = /root/breast_seg/nnUNet_Frame/dataset/nnUNet_preprocessed
nnUNet_results = /root/breast_seg/nnUNet_Frame/dataset/nnUNet_results


In [3]:
predict_single_image(
    image_path="/root/breast_seg/nnUNet_Frame/dataset/nnUNet_raw/Dataset001_BreastTumor/imagesTs_4mod/BreastMRI003_0000.nii.gz",
    output_mask_path="/root/breast_seg"
)


Running:
 
    nnUNetv2_predict     -i /tmp/nnunet_single_input     -o /tmp/nnunet_single_output     -d 1     -c 3d_fullres     -f 4     -chk checkpoint_best.pth
    

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting case_000:
perform_everything_on_device: True


  0%|          | 0/128 [00:00<?, ?it/s]


Prediction on device was unsuccessful, probably due to a lack of memory. Moving results arrays to CPU


  0%|          | 0/128 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/root/breast_seg/nnUNet_Frame/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 701, in predict_sliding_window_return_logits
    predicted_logits = self._internal_predict_sliding_window_return_logits(data, slicers,
  File "/venv/seg/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
  File "/root/breast_seg/nnUNet_Frame/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 665, in _internal_predict_sliding_window_return_logits
    raise e
  File "/root/breast_seg/nnUNet_Frame/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 644, in _internal_predict_sliding_window_return_logits
    prediction = self._internal_maybe_mirror_and_predict(workon)[0].to(results_device)
  File "/venv/seg/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
  File "/root/b

FileNotFoundError: Prediction not generated!